# Prepare MISR data

This notebook extracts radiometric and geometric information from the MISR data files.

Red BRF is taken from the MISR Georectified Radiance Product `MISR_AM1_GRP_ELLIPSOID_GM_Pxxx_Oxxxxxx_CC_F03_0024.hdf`.

Solar and viewing angles are taken from the geometric parameters ancillary dataset `MISR_AM1_GP_GMP_Pxxx_Oxxxxxx_F03_0013.hdf`.

xarray datasets are generated with coordinates represented in SOM, geodetic (lat/lon), and translated/rotated SOM coordinates. The translation/rotation occur such that the x and y axis are oriented with local east and north, respectively.

In [1]:
# conda env 'misr'
import MisrToolkit as mtk
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.MISR import hdf2xr, grp2xr

In [3]:
## Set data path
misr_data_dir = PROJECT_ROOT / 'data' / 'raw' / 'MISR'

## Specify MISR parameters
path = 117
orbit = 105254
cameras = ['DF', 'CF', 'BF', 'AF', 'AN', 'AA', 'BA', 'CA', 'DA']

## Define geographic region
center_lat = 19.266267063099104
center_lon = 119.72428415176724
region = mtk.MtkRegion(center_lat, center_lon, 8.8*3, 8.8*3, "km")

## Load MISR red band images
imagery = grp2xr(misr_data_dir, path, orbit, cameras, region)

## Load geometric parameters
gmp_file = misr_data_dir / f'MISR_AM1_GP_GMP_P{path}_O{orbit}_F03_0013.hdf'
field_list_solar = ['SolarZenith', 'SolarAzimuth']
field_list_vza = [f'{c.lower().capitalize()}Zenith' for c in cameras]
field_list_vaa = [f'{c.lower().capitalize()}Azimuth' for c in cameras]
field_list = field_list_solar + field_list_vza + field_list_vaa
gmp = hdf2xr(gmp_file, 'GeometricParameters', field_list, region)

In [7]:
imagery.to_netcdf(PROJECT_ROOT / 'data' / 'interim' / f'MISR_RAD_O{orbit:06}_{imagery.regionID}.nc')
gmp.to_netcdf(PROJECT_ROOT / 'data' / 'interim' / f'MISR_GEO_O{orbit:06}_{gmp.regionID}.nc')

In [ ]:
## Create AT3D coordinates
origin = gmp.isel(som_x=1, som_y=1)
X, Y = np.meshgrid(imagery.som_x, imagery.som_y, indexing='ij')
## 1. Translation setting origin to domain center
Xt = X - origin.som_x.data
Yt = Y - origin.som_y.data
# 2. Rotation about the origin orienting with north
Xtr = Xt * np.cos(origin.gamma.data) + Yt * np.sin(origin.gamma.data)
Ytr = -Xt * np.sin(origin.gamma.data) + Yt * np.cos(origin.gamma.data)
## 3. Translation setting corner of domain
Xtrt = Xtr + 8800
Ytrt = Ytr + 8800

imagery['x'] = (['som_x', 'som_y'], Xtrt)
imagery['y'] = (['som_x', 'som_y'], Ytrt)

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.scatter(np.ravel(imagery.x)/1000, np.ravel(imagery.y)/1000, c=np.ravel(imagery.isel(camera=4).brf), cmap='gray', vmin=0, vmax=0.4)
ax.set_xlabel('x')
ax.set_ylabel('y')
box_kwargs = {'ls':'--', 'lw':0.7, 'c':'r'}
ax.plot([0, 0], [0, 17.6], **box_kwargs)
ax.plot([0, 17.6], [0, 0], **box_kwargs)
ax.plot([17.6, 17.6], [0, 17.6], **box_kwargs)
ax.plot([0, 17.6], [17.6, 17.6], **box_kwargs)

In [ ]:
imagery.brf.plot(col='camera', cmap='gray', vmin=0, vmax=0.4)
plt.gca().invert_yaxis()